# 21. 🧰 Прикладное: рекомендательные системы

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/justxor/math-for-ai-2026/blob/main/notebooks/21_recommender_systems.ipynb)

Код из раздела [README.md](https://github.com/justxor/math-for-ai-2026/blob/main/README.md#m21). Запускайте ячейки сверху вниз: последующие используют переменные предыдущих. Теория, формулы и картинки — в тексте курса по ссылке.

In [ ]:
# Colab: всё нужное уже установлено. Локально: pip install -r requirements.txt
import numpy as np
import math
np.set_printoptions(precision=4, suppress=True)

### 🏋️ Практика модуля

**21.1.** Посчитайте NDCG@5 для выдачи с релевантностями [3, 2, 0, 1, 0] (идеальный порядок: [3, 2, 1, 0, 0]).

*▶️ Решение — сначала попробуйте сами.*

In [ ]:
import numpy as np
def dcg(rel): rel = np.asarray(rel, float); return np.sum((2**rel - 1) / np.log2(np.arange(2, len(rel) + 2)))
print(round(dcg([3, 2, 0, 1, 0]) / dcg([3, 2, 1, 0, 0]), 4))   # ≈ 0.9926

### 🏋️ Практика модуля

**21.2.** Реализуйте ALS для матрицы с пропусками и проверьте ошибку на скрытых значениях.

*▶️ Решение — сначала попробуйте сами.*

In [ ]:
rng = np.random.default_rng(5)
m, n, k, lam = 30, 40, 2, 0.1
R = rng.normal(size=(m, k)) @ rng.normal(size=(k, n)) + 0.3 * rng.normal(size=(m, n))
mask = rng.random((m, n)) < 0.3
r0 = np.random.default_rng(0); P = r0.normal(0, 0.1, (m, k)); Q = r0.normal(0, 0.1, (n, k))
for _ in range(100):
    for u in range(m):
        A = Q[mask[u]]; P[u] = np.linalg.solve(A.T @ A + lam * np.eye(k), A.T @ R[u, mask[u]])
    for i in range(n):
        A = P[mask[:, i]]; Q[i] = np.linalg.solve(A.T @ A + lam * np.eye(k), A.T @ R[mask[:, i], i])
print(np.sqrt(np.mean(((P @ Q.T) - R)[~mask] ** 2)))   # ≈ 0.39 при шуме 0.3

## 🔬 Лаборатория: иллюстрации модуля

Это код картинок из курса. Меняйте параметры (learning rate, температуру, число точек, ядро…) и перезапускайте ячейку — так интуиция появляется быстрее всего.

In [ ]:
# Помощники для иллюстраций (те же, что в scripts/make_figures.py)
import matplotlib.pyplot as plt
from scipy import stats
C = ["#2563eb", "#dc2626", "#059669", "#d97706", "#7c3aed", "#0891b2"]
plt.rcParams.update({"figure.dpi": 100, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.alpha": 0.25})

def save(fig, name):          # в ноутбуке просто показываем картинку
    fig.tight_layout(); plt.show()

def arrow(ax, v, color, label=None, origin=(0, 0), **kw):
    ax.annotate("", xy=(origin[0] + v[0], origin[1] + v[1]), xytext=origin,
                arrowprops=dict(arrowstyle="-|>", color=color, lw=2.2, **kw))
    if label:
        ax.text(origin[0] + v[0] * 1.08, origin[1] + v[1] * 1.08, label, color=color, fontsize=13, fontweight="bold")

def rosen_like(x, y):
    return 0.5 * x**2 + 4 * y**2

In [ ]:
def matrix_factorization():
    rng = np.random.default_rng(5)
    nu, ni, k = 30, 40, 2
    U = rng.normal(size=(nu, k)); V = rng.normal(size=(ni, k))
    R = U @ V.T + 0.3 * rng.normal(size=(nu, ni))
    mask = rng.random((nu, ni)) < 0.3
    Robs = np.where(mask, R, np.nan)
    # ALS
    r0 = np.random.default_rng(0)
    Uh = r0.normal(0, 0.1, (nu, k)); Vh = r0.normal(0, 0.1, (ni, k)); lam = 0.1
    for _ in range(100):
        for u in range(nu):
            m = mask[u]; A = Vh[m]; Uh[u] = np.linalg.solve(A.T @ A + lam * np.eye(k), A.T @ R[u, m])
        for i in range(ni):
            m = mask[:, i]; A = Uh[m]; Vh[i] = np.linalg.solve(A.T @ A + lam * np.eye(k), A.T @ R[m, i])
    P = Uh @ Vh.T
    rmse = np.sqrt(np.mean((P[~mask] - R[~mask]) ** 2))
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    vmin, vmax = -3, 3
    axes[0].imshow(Robs, cmap="RdBu_r", vmin=vmin, vmax=vmax); axes[0].set_title(f"Наблюдаемые оценки: {mask.mean():.0%} заполнено")
    axes[1].imshow(P, cmap="RdBu_r", vmin=vmin, vmax=vmax); axes[1].set_title(f"R̂ = U·Vᵀ (ранг {k}, ALS)\nRMSE на скрытых = {rmse:.2f} (шум 0.30)")
    axes[2].imshow(R, cmap="RdBu_r", vmin=vmin, vmax=vmax); axes[2].set_title("Истинные оценки (неизвестны)")
    for ax in axes:
        ax.set_xlabel("товары"); ax.set_ylabel("пользователи"); ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
    save(fig, "matrix_factorization")

matrix_factorization()